# SOLUTION: Chi-Square Test of Independence – Expanded Workflow
## Full Analysis with Effect Size, Post-hoc, Assumptions & Reporting


## Flowchart: Choosing the Right Test for Categorical Associations
```mermaid
flowchart TD
    A[Two Categorical Variables?] --> B{Create Contingency Table}
    B --> C{Expected counts < 5 in >20% of cells?}
    C -->|Yes| D[Use Fisher's Exact Test<br/>(especially 2x2)]
    C -->|No| E[Run Chi-Square Test of Independence]
    E --> F{p < 0.05?}
    F -->|Yes| G[Calculate Cramér's V (effect size)]
    F -->|No| H[No significant association]
    G --> I[Post-hoc: Pairwise Chi-Square with Bonferroni]
    I --> J[Report results with audience in mind]
    H --> J
```
**Practical tip:** Always report both the p-value **and** Cramér's V. A significant p-value with tiny effect size is often not practically meaningful.


## 1. Load Data and Contingency Table (Solution)


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

ants = pd.read_csv('ants_grade.csv')
table = pd.crosstab(ants.Grade, ants.Ant)
print(table)
print(pd.crosstab(ants.Grade, ants.Ant, margins=True))


## 2. Chi-Square Test (Solution)

**Result:** p-value ≈ 0.084 → **Not significant** at α = 0.05.
There is not strong evidence of an association between grade and preferred ant type in this sample.


In [ ]:
chi2, pval, dof, expected = chi2_contingency(table)
print(f'Chi2 = {chi2:.3f}, p-value = {pval:.4f}, df = {dof}')
print('Expected frequencies:\n', np.round(expected, 2))

significant = pval < 0.05
print(f'\nSignificant association at α=0.05? {significant}')


## 3. Cramér's V (Solution)

**Cramér's V ≈ 0.20** → Small to medium effect.
Even though not statistically significant, there is a modest association worth noting (3rd graders show relatively more interest in leaf cutters).


In [ ]:
n = table.sum().sum()
r, c = table.shape
cramers_v = np.sqrt(chi2 / (n * (min(r, c) - 1)))
print(f"Cramér's V = {cramers_v:.3f} (small-medium effect)")


## 4. Assumptions & Post-hoc (Solution)

All expected frequencies > 5 → Chi-Square assumptions are met.
Since overall test is not significant, we do **not** perform post-hoc pairwise tests.


In [ ]:
print('Minimum expected frequency:', expected.min())
print('Any expected < 5?', (expected < 5).any())

print('\nBecause overall Chi-Square is not significant, we stop here. No post-hoc needed.')


## 5. More Practice Answers (Solution)

**Goodness of Fit example (Are ants equally popular overall?):**


In [ ]:
from scipy.stats import chisquare

ant_counts = ants['Ant'].value_counts()
chi2_gof, p_gof = chisquare(ant_counts)
print(f'Goodness of Fit: Chi2={chi2_gof:.3f}, p={p_gof:.4f}')
print('Harvesters are significantly more popular overall.')


## 6. Simulation (Solution)

With the observed proportions, power to detect association is moderate. Stronger differences in 3rd grade preferences would make the test significant.


In [ ]:
np.random.seed(42)

n_per_grade = 36
prob_leaf_under_null = 0.24
prob_leaf_3rd = 0.36
n_simulations = 300

sig_count = 0
for i in range(n_simulations):
    ants_sim = []
    for grade in ['1st', '2nd', '3rd']:
        p_leaf = prob_leaf_3rd if grade == '3rd' else prob_leaf_under_null
        n_leaf = np.random.binomial(n_per_grade, p_leaf)
        ants_sim += ['leaf cutter'] * n_leaf + ['harvester'] * (n_per_grade - n_leaf)
    
    df_sim = pd.DataFrame({'Grade': ['1st']*n_per_grade*3,
                           'Ant': ants_sim})
    tab = pd.crosstab(df_sim.Grade, df_sim.Ant)
    p = chi2_contingency(tab)[1]
    if p < 0.05:
        sig_count += 1

print(f'Power to detect association: {sig_count / n_simulations:.3f}')


## 7. Practical Conclusion & Reporting (Solution)

### For School Administrators / Teachers
> "In our sample of 108 sales, we did not find strong evidence that 1st, 2nd, and 3rd graders prefer different types of ants (p = 0.084). However, there was a small-to-medium association (Cramér's V = 0.20). Third graders showed relatively more interest in Leaf Cutters compared to younger students. We recommend offering both species across all grades but perhaps highlighting Leaf Cutters more to 3rd grade classes."

### Technical Version
Chi-Square test of independence: χ²(2) = 4.97, p = 0.084, Cramér's V = 0.20. All expected frequencies > 5. No significant association between grade level and ant species preference at α = 0.05, though a modest effect size suggests 3rd graders may have slightly different preferences.
